# CCE Proof-of-Concept: V6 + CCE-as-feature ablation

Runs the 6-arm ablation defined in `research/paper/runner/cce_features.py::ABLATION_ARMS` on V6's 20 balanced tasks.

**Decision the PoC answers:** does adding contrastive-code-entropy features (H_code − H_lang) to V6's hidden-state-based hallucination detector carry signal that the V6 features alone don't already capture? If `v6_plus_cce` beats `v6_full` by a meaningful margin, the paper has a defensible novelty story against SAPLMA / FLARE / DRAGIN. If they're within CIs, V6 alone is not differentiated enough.

**Compute:** Colab Pro (A100 or V100). ~1 hour wall-clock.

**Output:** `research/paper/cce_poc_results.json`, plus printed tables for both Phase 2 (LOO classifier ablation) and Phase 3 (end-to-end retrieval gating).

## 1. Install dependencies

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers scikit-learn

## 2. Clone repo and httpx

In [ ]:
import os
if not os.path.exists('/content/reposynth'):
    !git clone https://github.com/aniJani/reposynth.git /content/reposynth
%cd /content/reposynth
!git checkout Research
!git pull --rebase || true

if not os.path.exists('/content/httpx'):
    !git clone --depth 1 https://github.com/encode/httpx.git /content/httpx

## 3. HuggingFace login (CodeLlama is gated)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
try:
    login(userdata.get('HF_TOKEN'))
except Exception as e:
    print('Set HF_TOKEN as a Colab secret first:', e)

## 4. Mount Drive (so results survive a Colab disconnect)

Optional but recommended. Saves `cce_poc_results.json` and `cce_poc_features.json` to Drive so you can resume Phase 2/3 without redoing Phase 1's GPU generation.

In [ ]:
MOUNT_DRIVE = True
RESULTS_PATH = 'research/paper/cce_poc_results.json'
CACHE_PATH   = 'research/paper/cce_poc_features.json'

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = '/content/drive/MyDrive/cce_poc'
    !mkdir -p {drive_dir}
    RESULTS_PATH = f'{drive_dir}/cce_poc_results.json'
    CACHE_PATH   = f'{drive_dir}/cce_poc_features.json'
    print(f'Will write to {RESULTS_PATH}')

## 5. Run the PoC

Phase 1 (feature extraction) is the long step: ~20 generations × ~10–30s each = ~10 minutes on A100. Phase 2 (LOO ablation) is seconds. Phase 3 (per-arm retrieval gating) is ~6–12 more minutes depending on how many arms predict errors.

If Phase 1 has already completed and `cce_poc_features.json` exists, pass `--skip-generation` to redo just the ablation analysis (instant).

In [ ]:
!python research/paper/cce_poc.py \
    --httpx-dir /content/httpx \
    --out {RESULTS_PATH} \
    --features-cache {CACHE_PATH}

## 6. Inspect the ablation table

The headline question: does `v6_plus_cce` (the full feature set) beat `v6_full` (V6's original 11) on Phase 2 F1 and Phase 3 final accuracy?

In [ ]:
import json
with open(RESULTS_PATH) as f:
    res = json.load(f)

print('PHASE 2: classifier discriminative power (LOO CV)')
print(f"{'arm':<20} {'#feat':>5} {'acc':>5} {'prec':>5} {'rec':>5} {'f1':>5}")
for arm, r in res['phase2_loo_ablation'].items():
    print(f"{arm:<20} {r['n_features']:>5d} {r['accuracy']:>5.3f} "
          f"{r['precision']:>5.3f} {r['recall']:>5.3f} {r['f1']:>5.3f}")

if res['phase3_end_to_end']:
    print()
    print('PHASE 3: end-to-end with retrieval gating')
    print(f"{'arm':<20} {'init':>5} {'final':>5} {'always':>5} {'used':>5} {'saved':>5} {'save%':>6}")
    for arm, r in res['phase3_end_to_end'].items():
        print(f"{arm:<20} {r['initial_accuracy']:>5.3f} {r['final_accuracy']:>5.3f} "
              f"{r['always_retrieve_accuracy']:>5.3f} {r['n_retrievals_used']:>5d} "
              f"{r['n_retrievals_saved']:>5d} {r['retrieval_save_rate']*100:>5.1f}%")

# Headline comparison
p2 = res['phase2_loo_ablation']
delta_f1 = p2['v6_plus_cce']['f1'] - p2['v6_full']['f1']
delta_acc = p2['v6_plus_cce']['accuracy'] - p2['v6_full']['accuracy']
print()
print(f'HEADLINE Δ (v6_plus_cce − v6_full):  Δacc={delta_acc:+.3f}  Δf1={delta_f1:+.3f}')
if delta_f1 > 0.05:
    print('  → CCE features carry signal. The paper has a novelty story.')
elif delta_f1 > 0.01:
    print('  → CCE features add small signal. Need n>20 with CIs to declare significance.')
else:
    print('  → CCE features do NOT add signal beyond V6. Paper has to pivot.')

## 7. Next steps based on the result

| Outcome | What to do |
|---|---|
| `v6_plus_cce` clearly beats `v6_full` (Δf1 > 0.05) | Scale to v2 benchmark (≥100 questions across Flask/FastAPI/Requests/httpx). The paper-novelty story is locked. |
| Marginal improvement (Δf1 0.01–0.05) | Same scale-up, but expect modest gains. Workshop tier most likely. |
| No improvement (Δf1 ≤ 0.01) | The H_code−H_lang work isn't pulling its weight on top of V6. Either (a) re-pitch the paper purely as a SAPLMA-for-code / hallucination-detection-for-code-Q&A contribution and accept the related-work crowd, or (b) abandon V6 and pivot to the cleaner training-free `cce_only` approach if its results are non-trivial. |